In [ ]:
# --- ENVIRONMENT SETUP: Environment-Aware Paths ---
import sys, os
from pathlib import Path

try:
    # Standard Colab setup
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = Path(str(REPO_ROOT))
except ImportError:
    # Local fallback (assumes running from explorations/ or experiments/)
    cur = Path().resolve()
    REPO_ROOT = cur.parent if cur.name in ('explorations', 'experiments') else cur.parents[1]

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# --- Standalone Environment Configuration ---
import os
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
    DRIVE_ROOT = Path('/content/drive/MyDrive')
except ImportError:
    IN_COLAB = False
    DRIVE_ROOT = Path(os.environ.get('DRIVE_ROOT', 'G:/My Drive'))

DATA_LAKE = DRIVE_ROOT / 'data_lake'
BRONZE = DATA_LAKE / '01_bronze_medical'
SILVER = DATA_LAKE / '02_silver'
GOLD = DATA_LAKE / '03_gold'
MODELS = DRIVE_ROOT / 'models'
PRETRAINED = MODELS / 'pretrained'
TRAINED = MODELS / 'trained'
OPS = DRIVE_ROOT / 'ops'
MLFLOW_TRACKING_URI = f'file://{OPS / "mlflow" / "mlruns"}'
REPOS = DRIVE_ROOT / 'repos'


# 🔬 EDA & Exploration: Polyp Instance Segmentation

Self-contained experimentation environment for YOLO11-seg polyp detection.

**Goal**: Visually explore the dataset and conduct hyperparameter sweeping to find optimal training configurations.

In [ ]:
# === 1. Setup Environment ===
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics "ray[tune]" seaborn

import os
import glob
import matplotlib.pyplot as plt
from ultralytics import YOLO
import torch

DATASET_YAML = str(GOLD / '016_polyp_fast_diag_dataset/dataset.yaml')
if not Path(DATASET_YAML).exists():
    raise FileNotFoundError(
        f"❌ Dataset not found at: {DATASET_YAML}\n"
        "We rely on the Medallion Data Lake architecture.\n"
        "1. Create a `.env` file at the root of the repository.\n"
        "2. Set `DATA_LAKE_DIR=/path/to/your/data_lake`.\n"
        "3. Ensure the dataset is located at `{DATA_LAKE_DIR}/03_gold/...`"
    )


## 2. Visualize the Ground Truth Data
Explore the annotation mapping visually to ensure polygons match up.

In [ ]:
from IPython.display import Image, display
import random
import glob
from pathlib import Path

val_images = glob.glob(str(Path(DATASET_YAML).parent / 'val' / 'images' / '*.*'))
if val_images:
    sample = random.choice(val_images)
    print(f"Sample Ground Truth Target: {sample}")
    display(Image(filename=sample, width=400))


## 3. Hyperparameter Sweep (Ray Tune)
Use YOLO's native `tune()` integration to sweep through Learning Rates, momentums, optimizer constraints, and augmentations.
**Tweak instructions:** Adjust `iterations` to set how many configurations to search, and `epochs` for the depth of each search.

In [ ]:
try:
    import ray
    from ray import tune
except ImportError:
    print("Please install ray[tune] first")
    raise

model = YOLO('yolo11n-seg.pt')

print("Starting tuning run...")
results = model.tune(
    data=DATASET_YAML,
    use_ray=True, 
    iterations=5,      # <-- TWEAK THIS: How many different hyperparam configurations to try
    epochs=15,          # <-- TWEAK THIS: Training epochs per iteration
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    project="/content/drive/MyDrive/models/tuning",
    name="016_polyp_tune"
)


## 4. Plotting Tuning Results
Extract scatter plots and evaluation charts from the Ray Tune CSV to evaluate performance.

In [ ]:
import pandas as pd
import seaborn as sns
import glob

results_paths = glob.glob(str(MODELS / 'tuning' / '016_polyp_tune/**/tune_results.csv'), recursive=True)

if results_paths:
    results_paths.sort(key=os.path.getmtime, reverse=True)
    results_df = pd.read_csv(results_paths[0])
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Safe column access
    lr_col = 'lr0' if 'lr0' in results_df.columns else results_df.columns[0]
    map_b_col = 'metrics/mAP50-95(B)' if 'metrics/mAP50-95(B)' in results_df.columns else results_df.columns[1]
    map_m_col = 'metrics/mAP50-95(M)' if 'metrics/mAP50-95(M)' in results_df.columns else results_df.columns[2]
    wd_col = 'weight_decay' if 'weight_decay' in results_df.columns else results_df.columns[3]
    mom_col = 'momentum' if 'momentum' in results_df.columns else None
    
    sns.scatterplot(data=results_df, x=lr_col, y=map_b_col, ax=ax1, hue=mom_col)
    ax1.set_title('Learning Rate vs BBox mAP')
    
    sns.scatterplot(data=results_df, x=wd_col, y=map_m_col, ax=ax2, hue=lr_col)
    ax2.set_title('Weight Decay vs Segmentation mAP')
    
    plt.show()
else:
    print("Results file not found. Did the tuning job finish?")


## 5. Visualizing Predictions vs Ground Truth
After tuning, load the best model checkpoint and visualize it against validation images.

In [ ]:
# WARNING: Run this ONLY after you have a tuned model in your tuning directory!
import os
import random
import matplotlib.pyplot as plt
import glob
from ultralytics import YOLO
from pathlib import Path

best_model_paths = glob.glob(str(MODELS / 'tuning' / '016_polyp_tune/**/weights/best.pt'), recursive=True)
if len(best_model_paths) > 0:
    best_model_paths.sort(key=os.path.getmtime, reverse=True)
    top_model = YOLO(best_model_paths[0])
    
    all_imgs = glob.glob(str(Path(DATASET_YAML).parent / 'val' / 'images' / '*'))
    if all_imgs:
        test_images = random.sample(all_imgs, min(3, len(all_imgs)))
        for img_path in test_images:
            results = top_model(img_path)
            res_plotted = results[0].plot()
            plt.figure(figsize=(10, 5))
            plt.imshow(res_plotted[..., ::-1])
            plt.title(f'Predictions: {os.path.basename(img_path)}')
            plt.axis('off')
            plt.show()
else:
    print("No trained model checkpoints found. Did you run the tune cell?")


## 6. Deep Learning EDA: Object Size Analysis
Understanding your bounding box geometries helps guide anchor configurations and image augmentations.

In [ ]:
from IPython.display import Image, display
import random
import glob
from pathlib import Path

val_images = glob.glob(str(Path(DATASET_YAML).parent / 'val' / 'images' / '*.*'))
if val_images:
    sample = random.choice(val_images)
    print(f"Sample Ground Truth Target: {sample}")
    display(Image(filename=sample, width=400))


## 7. Interactive Out-of-Distribution Test
Upload your own local medical image to instantly simulate zero-shot inference.

In [ ]:
from google.colab import files
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import glob
import os

print("Upload a colonoscopy image to test the model (e.g., .jpg or .png)")
uploaded = files.upload()

best_model_paths = glob.glob(str(MODELS / 'tuning' / '016_polyp_tune/**/weights/best.pt'), recursive=True)
if len(best_model_paths) > 0 and uploaded:
    best_model_paths.sort(key=os.path.getmtime, reverse=True)
    model = YOLO(best_model_paths[0])
    
    for filename in uploaded.keys():
        results = model(filename)
        res_plotted = results[0].plot()
        
        plt.figure(figsize=(10, 10))
        plt.imshow(res_plotted[..., ::-1])
        plt.title('Zero-Shot Inference')
        plt.axis('off')
        plt.show()
elif not uploaded:
    print("Waiting for upload...")
else:
    print("Please verify your tuned model checkpoint exists before running inference.")


## 8. Training vs Validation Loss Curves
Plot the training curves over epochs for the best trial to check for overfitting.

In [ ]:
import pandas as pd
import glob
import matplotlib.pyplot as plt
import os

trial_csvs = glob.glob(str(MODELS / 'tuning' / '016_polyp_tune/**/results.csv'), recursive=True)
if trial_csvs:
    trial_csvs.sort(key=os.path.getmtime, reverse=True)
    df = pd.read_csv(trial_csvs[0])
    df.columns = df.columns.str.strip()
    
    if 'epoch' in df.columns and 'train/box_loss' in df.columns and 'val/box_loss' in df.columns:
        plt.figure(figsize=(10, 5))
        plt.plot(df['epoch'], df['train/box_loss'], label='Train Box Loss', marker='o')
        plt.plot(df['epoch'], df['val/box_loss'], label='Val Box Loss', marker='s')
        
        if 'train/seg_loss' in df.columns and 'val/seg_loss' in df.columns:
            plt.plot(df['epoch'], df['train/seg_loss'], label='Train Seg Loss', marker='x', linestyle='--')
            plt.plot(df['epoch'], df['val/seg_loss'], label='Val Seg Loss', marker='+', linestyle='--')
            
        plt.title('Train vs Validation Loss (Best Trial)')
        plt.xlabel('Epochs')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True)
        plt.show()
    else:
        print("Columns missing in results.csv")
else:
    print("No trial results.csv found. Plotting requires a completed tune or training iteration.")


In [ ]:
# Release resources
from google.colab import runtime
runtime.unassign()
